In [1]:
import time
notebook_start = time.perf_counter()

%pip install -e /home/darshan/A6/PCSAFT_cDFT/thermoift

import feos
import si_units as si
import numpy as np
import importlib.metadata
from pathlib import Path
from itertools import combinations

# Import thermoift modules
import thermoift.PLOT_SETTINGS as ps
from thermoift import KIJ
from thermoift.FeosPlugin import (
    RegistryManager,
    CompositionHandler,
    ParameterBuilder,
    PropertyCalculator,
    VLECalculator,
    InterfacialTensionCalculator,
    DataProcessor,
    PlottingEngine
)
from thermoift.semi_emperical_correlations import semi_emperical_correlations

print(f"FEOS version used: {importlib.metadata.version('feos')}")
KIJ_DIR = Path(KIJ.__file__).parent

Obtaining file:///home/darshan/A6/PCSAFT_cDFT/thermoift
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for thermoift (pyproject.toml) ... done
  Created wheel for thermoift: filename=thermoift-0.2.0-0.editable-py3-none-any.whl size=1730 sha256=8faa56f41dfc7599aea2c74e02eb0b214d21f016e570197b53ee02bef2762b2f
  Stored in directory: /tmp/pip-ephem-wheel-cache-175vttct/wheels/fd/2f/c4/54a2ee5cd16a9bf5b183bbe5c28d1b3ba4926fb0261a13e1e4
Successfully built thermoift
  Attempting uninstall: thermoift
    Found existing installation: thermoift 0.2.0
    Uninstalling thermoift-0.2.0:
      Successfully uninstalled thermoift-0.2.0
Note: you may need to restart the kernel to use updated packages.
FEOS version used: 0.9.4


## Cell 2: Setup - Components, KIJ Models, Feeds, and Settings

In [2]:
COMPONENTS  = ["carbon dioxide", "hydrogen", "argon"]
# KIJ_map     = {
#                 ("CO2", "Ar")   : "constant",
#                 ("H2", "Ar")    : "linear",
#                 ("CO2", "H2")   : "linear"}

KIJ_map     = {
                ("CO2", "Ar")   : "constant",
                ("H2", "Ar")    : "zero",
                ("CO2", "H2")   : "zero"}

CO2_comp    = 0.99
n_feeds     = 4
TEST_RUN    = True
verbose     = False
CSV_FOLDER  = "CSV"

In [ ]:
if TEST_RUN == True:
    T_initial   = 200
    P_TOL       = 1e-2          # Pressure [bar] tolerance for flash calculations
    T_STEP      = 100           # Number of T grid points for bubble/dew curves
    P_STEP      = 5             # Pressure step for flash calculations [bar]
    lgrid       = 100           # Length of computational domain [Angstrom]
    ngrid       = 500           # Number of grid points for planar interface
    PT_results  = {}            # Store phase envelope results
    VLE_DFT     = {}            # Store VLE-DFT combined results

elif TEST_RUN == False:
    T_initial   = 200
    P_TOL       = 1e-4
    T_STEP      = 500
    P_STEP      = 1
    lgrid       = 100
    ngrid       = 2048
    PT_results  = {}
    VLE_DFT     = {}

feeds       = CompositionHandler.generate_feeds(CO2=CO2_comp, n_points=n_feeds)
builder     = KIJ.KIJMatrixBuilder(root=str(KIJ_DIR), kij_filename="KIJ.json", verbose=False)
parameters  = ParameterBuilder.build_parameters(COMPONENTS, T_K=300.0, kij_builder=builder, model_map=KIJ_map)
components  = RegistryManager.get_component_names(parameters)
KIJ_LABELS  = RegistryManager.kij_labels_from_names(COMPONENTS)

builder.build_and_display(components=KIJ_LABELS, model_map=KIJ_map, T_eval=300.0,
    show_matrix=True, show_pair_equations=True)
CompositionHandler.print_array2d(feeds)
builder.show_pair_plots(KIJ_LABELS, mode="all")

parameters

## Cell 4: Phase Envelope Calculations (Bubble/Dew Curves)

In [ ]:
# Phase envelope with T-dependent kij, continuation, and non-uniform T grid

for k, z in enumerate(feeds, start=1):
    z    = CompositionHandler.normalize_z(z)
    feed = CompositionHandler.compute_feed_moles(z)

    if verbose:
        print(f"\nFeed {k}: z = {z}")

    active_z, active_components, is_reduced = CompositionHandler.reduce_components(
        z, COMPONENTS, verbose=verbose)
    active_feed = CompositionHandler.compute_feed_moles(active_z)
    active_map  = ParameterBuilder.reduce_kij_map(active_components, KIJ_map)

    parameters_ref = ParameterBuilder.build_parameters(
        active_components, T_K=T_initial, kij_builder=builder, model_map=active_map)
    eos = feos.HelmholtzEnergyFunctional.pcsaft(parameters_ref)

    try:
        CT, CP = VLECalculator.compute_critical_point(eos, active_z, T_guess=T_initial)

        if verbose:
            print(f"Critical Point: T = {CT/si.KELVIN:.2f} K, P = {CP/si.BAR:.2f} bar")

        Tc_K     = float(CT / si.KELVIN)
        eos_fn   = VLECalculator.make_eos_factory(active_components, builder, active_map)
        T_values = VLECalculator.make_T_grid(T_initial, Tc_K, T_STEP)

        T_bubble_all, P_bubble_all, T_dew_all, P_dew_all = VLECalculator.compute_phase_envelope(
            eos_fn, T_values, active_feed, verbose, Tc=Tc_K)

        PT_results[f"feed_{k}"] = {
            "z":      [round(float(x), 6) for x in np.asarray(z, dtype=float)],
            "TC_K":   round(float(CT / si.KELVIN), 2),
            "PC_bar": round(float(CP / si.BAR), 2),
            "bubble": [{"T_K": float(T), "P_bar": float(P)} for T, P in zip(T_bubble_all, P_bubble_all)],
            "dew":    [{"T_K": float(T), "P_bar": float(P)} for T, P in zip(T_dew_all, P_dew_all)],
        }

    except Exception as e:
        print(f"Feed {k}: z = {z}")
        print("Critical point computation failed:", e)

## Cell 5: PT Diagram Plot, TP Flash, and cDFT Calculations

In [ ]:
# PT diagram plots and phase envelope data initialization

PT_figs  = {}
VLE_DFT  = {}
_pt_data = {}  # internal: T/P flash grid per feed

for feed_key, feed in PT_results.items():
    TP_z = np.array(feed.get("z", []))
    CT   = feed["TC_K"] * si.KELVIN
    CP   = feed["PC_bar"] * si.BAR

    fig, ax = PlottingEngine.plot_phase_diagram(PT_results, feed_key, parameters)
    PT_figs[feed_key] = (fig, ax)

    bub_pts   = sorted(feed.get("bubble", []), key=lambda p: p["T_K"])
    dew_pts   = sorted(feed.get("dew",    []), key=lambda p: p["T_K"])
    T_bub_arr = np.array([p["T_K"]   for p in bub_pts])
    P_bub_arr = np.array([p["P_bar"] for p in bub_pts])
    T_dew_arr = np.array([p["T_K"]   for p in dew_pts])
    P_dew_arr = np.array([p["P_bar"] for p in dew_pts])

    # T range covered by both curves; bubble T values, dew pressure via interpolation
    T_lo     = max(T_bub_arr.min(), T_dew_arr.min())
    T_hi     = min(T_bub_arr.max(), T_dew_arr.max()) * 0.999
    mask     = (T_bub_arr >= T_lo) & (T_bub_arr <= T_hi)
    common_T = T_bub_arr[mask]
    P_bub_at = P_bub_arr[mask]
    P_dew_at = np.interp(common_T, T_dew_arr, P_dew_arr)

    VLE_DFT[feed_key] = {
        "z": TP_z.tolist(),
        "phase_envelope": {
            "bubble_curve": [{"T_K": float(T), "P_bar": float(P)} for T, P in zip(T_bub_arr, P_bub_arr)],
            "dew_curve":    [{"T_K": float(T), "P_bar": float(P)} for T, P in zip(T_dew_arr, P_dew_arr)],
            "isothermal_lines": [
                {"T_K": float(T), "P_bubble_bar": float(Pb), "P_dew_bar": float(Pd)}
                for T, Pb, Pd in zip(common_T, P_bub_at, P_dew_at) if Pb > Pd
            ]
        },
        "interfacial_data": {}
    }
    _pt_data[feed_key] = {
        "CT": CT, "CP": CP, "TP_z": TP_z,
        "common_T": common_T, "P_bub_at": P_bub_at, "P_dew_at": P_dew_at,
    }

    if verbose:
        print(f"\n{feed_key}: {len(common_T)} isothermal flash temperatures")

In [ ]:
# TP flash and cDFT interfacial tension calculations

sec = semi_emperical_correlations(n_grid=ngrid, l_grid=lgrid)

for feed_key, feed in PT_results.items():
    fig, ax  = PT_figs[feed_key]
    meta     = _pt_data[feed_key]
    CT, CP   = meta["CT"], meta["CP"]
    TP_z     = meta["TP_z"]
    common_T = meta["common_T"]
    P_bub_at = meta["P_bub_at"]
    P_dew_at = meta["P_dew_at"]

    active_z, active_components, is_reduced = CompositionHandler.reduce_components(
        TP_z, COMPONENTS, verbose=verbose)
    active_feed = CompositionHandler.compute_feed_moles(active_z)
    active_map  = ParameterBuilder.reduce_kij_map(active_components, KIJ_map)

    if verbose:
        print(f"\n{feed_key}:")

    for T_K, P_b, P_d in zip(common_T, P_bub_at, P_dew_at):
        if P_b <= P_d:
            continue

        parameters_T = ParameterBuilder.build_parameters(
            active_components, T_K=float(T_K), kij_builder=builder, model_map=active_map)
        eos_T        = feos.HelmholtzEnergyFunctional.pcsaft(parameters_T)
        molar_masses = PropertyCalculator.molar_masses(parameters_T)

        gamma0_CO2, rhoL0_CO2, rhoV0_CO2, Tc_CO2, Pc_CO2, Psat_CO2 = \
            sec._pure_component_cDFT("carbon dioxide", float(T_K))

        pressures = np.linspace(P_d + P_TOL, P_b - P_TOL, 10)
        ax.plot([T_K, T_K], [P_d, P_b], linewidth=0.5, linestyle="--", color="k", alpha=0.3)
        VLE_DFT[feed_key]["interfacial_data"][T_K] = []

        if verbose:
            print(f"  T = {T_K:.1f} K: P_bub = {P_b:.2f}, P_dew = {P_d:.2f} bar")

        for P in pressures:
            if verbose:
                print(f"T = {T_K:.1f} K | P_flash = {P:.2f} | z = {active_z}")

            try:
                eq, x, y, liquid_density, vapor_density = VLECalculator.tp_flash(
                    eos_T, T_K*si.KELVIN, P*si.BAR, active_z*si.MOL, molar_masses)
            except Exception as e:
                print(f"TP flash failed for z = {active_z} at T = {T_K:.1f} K, P = {P:.2f} bar: {e}")
                continue

            interface = InterfacialTensionCalculator.build_planar_interface(
                eq, critical_temperature=CT, n_grid=ngrid, l_grid=lgrid)

            try:
                gamma_mN_m, interfacial_thickness_nm, enrichment = \
                    InterfacialTensionCalculator.solve_interface_properties(interface)
            except Exception as e:
                print(f"Surface tension calculation failed for z = {active_z} at T = {T_K:.1f} K, P = {P:.2f} bar: {e}")
                gamma_mN_m            = np.nan
                interfacial_thickness_nm = np.nan
                enrichment            = tuple(np.nan for _ in active_components)

            if verbose:
                print(f"T = {T_K:.1f} K | P_flash = {P:.2f} bar | gamma = {gamma_mN_m:.6f} mN/m")

            row_data = DataProcessor.assemble_row(
                T_K, P, active_z, CT, CP, P_b, P_d, liquid_density, vapor_density,
                x, y, gamma_mN_m, interfacial_thickness_nm, active_components, enrichment,
                gamma0_CO2=gamma0_CO2,
                rhoL0_CO2_mol_cm3=rhoL0_CO2,
                rhoV0_CO2_mol_cm3=rhoV0_CO2,
                Tc_CO2=Tc_CO2,
                Psat_CO2=Psat_CO2,
                ML_mode=True)

            VLE_DFT[feed_key]["interfacial_data"][T_K].append(row_data)

    PlottingEngine.save_plots(fig, f"PT_{feed_key}")

    fig_gamma = PlottingEngine.plot_interfacial_tension_map(PT_results, VLE_DFT, feed_key, parameters)
    if fig_gamma is not None:
        PlottingEngine.save_plots(fig_gamma, f"Gamma_{feed_key}")

summary = DataProcessor.summarize_vle_dft(VLE_DFT)
if verbose:
    print(summary)

saved_files = DataProcessor.export_to_csv(VLE_DFT, folder=CSV_FOLDER, verbose=verbose)

In [7]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")

Total notebook runtime: 0.75 minutes
